# Step 10 — Majority-class baseline

This notebook establishes the minimum benchmark for the final model comparison. The last prediction date is November 22, 2024.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "aapl_momentum_sentiment.csv"
CUTOFF_DATE = pd.Timestamp("2024-11-22")
data = pd.read_csv(DATA_PATH, parse_dates=["date"], index_col="date").sort_index()
data = data.loc[data.index <= CUTOFF_DATE].copy()
print(f"Eligible observations through {CUTOFF_DATE.date()}: {len(data):,}")

Eligible observations through 2024-11-22: 472


## Chronological split

We reserve the latest 25% of eligible dates for testing. Five dates immediately before testing form a safety gap because every target looks five trading days into the future.

In [2]:
test_size = int(len(data) * 0.25)
gap_size = 5
test_start = len(data) - test_size

train = data.iloc[: test_start - gap_size].copy()
safety_gap = data.iloc[test_start - gap_size : test_start].copy()
test = data.iloc[test_start:].copy()

pd.DataFrame({
    "Rows": [len(train), len(safety_gap), len(test)],
    "Start": [train.index.min(), safety_gap.index.min(), test.index.min()],
    "End": [train.index.max(), safety_gap.index.max(), test.index.max()],
}, index=["Training", "Safety gap", "Testing"])

,Rows,Start,End
Training,349,2023-01-10,2024-05-30
Safety gap,5,2024-05-31,2024-06-06
Testing,118,2024-06-07,2024-11-22


## Find the training majority

Only training outcomes determine the majority class. Test outcomes remain unused until the predictions are complete.

In [3]:
training_counts = train["outperformed"].astype(int).value_counts().sort_index()
training_rates = train["outperformed"].astype(int).value_counts(normalize=True).sort_index()
training_distribution = pd.DataFrame({
    "Observations": training_counts,
    "Percentage": training_rates * 100,
}).rename(index={0: "Did not outperform", 1: "Outperformed"})
majority_class = int(train["outperformed"].mode().iloc[0])
print("Majority prediction:", "Outperform" if majority_class == 1 else "Do not outperform")
training_distribution.round(2)

Majority prediction: Outperform


,Observations,Percentage
outperformed,,
Did not outperform,165,47.28
Outperformed,184,52.72


## Apply the rule to unseen dates

The baseline makes exactly the same prediction for every test date.

In [4]:
actual = test["outperformed"].astype(int)
predicted = pd.Series(majority_class, index=test.index, name="prediction")

metrics = pd.Series({
    "Accuracy": accuracy_score(actual, predicted),
    "Precision": precision_score(actual, predicted, zero_division=0),
    "Recall": recall_score(actual, predicted, zero_division=0),
    "Correct predictions": int(actual.eq(predicted).sum()),
    "Test observations": len(actual),
})
metrics

Accuracy                 0.5
Precision                0.5
Recall                   1.0
Correct predictions     59.0
Test observations      118.0
dtype: float64

In [5]:
pd.DataFrame({
    "Actual outcomes": actual.value_counts().sort_index(),
    "Percentage": actual.value_counts(normalize=True).sort_index() * 100,
}).rename(index={0: "Did not outperform", 1: "Outperformed"}).round(2)

,Actual outcomes,Percentage
outperformed,,
Did not outperform,59,50.0
Outperformed,59,50.0


## Interpretation

This baseline does not use momentum or sentiment. Its accuracy is the threshold the feature-based models should improve upon on these exact same test dates.